# Pemodelan — Random Forest Kuantil

## 1. Konfigurasi

In [ ]:
# Notebook ini menjalankan satu model kandidat dari awal sampai akhir, dalam empat tahap
# berurutan: benchmark (§3), pencarian hyperparameter (§4), walk-forward lima fold (§5), dan
# model final (§6) — ditutup ringkasan hasil (§7). Desember 2025 terkunci sebagai test set dan
# tidak dinilai di sini.
#
# Kode modelnya ada di §2 notebook ini, salinan verbatim dari
# utils/modelling/model_random_forest.py; §8 membandingkan keduanya dan menyebut fungsi mana
# yang menyimpang kalau ada. Mesin bersama yang dipakai ketiga notebook modeling tetap diimpor
# dari utils — modeling_prep, walk_forward, evaluation, model_common, purging, run_config —
# karena menyalin 2.129 baris itu ke tiga notebook lebih mahal daripada pemisahan yang
# dibelinya (keputusan pemilik proyek 2026-08-26).
#
# Output tersimpan di §4-§8 berasal dari run Fase 3 Random Forest **2026-08-25** (K1 = 2,8621
# lima fold, 2,8508 di potongan bersih fold 1/2/4), dipindahkan dari notebook versi pra-refactor
# — kodenya identik selain prefiks `rf.` yang hilang ketika modul diinline ke §2. Output §3 tidak
# ikut dibawa karena selnya bertambah dua baris print. Bukti lengkapnya hidup di git, bukan di
# sel: docs/hasil-modeling-rf.md, dataset/model_ready/rf_{search,walk_forward}_results.csv,
# rf_best_params.json, dan models/random_forest_q90.joblib.
#
# Desain: docs/superpowers/specs/2026-08-18-random-forest-modeling-design.md
# Rencana: docs/superpowers/plans/2026-08-18-random-forest-modeling.md
# Hasil terukur: docs/hasil-modeling-rf.md
import sys
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np
import pandas as pd
from quantile_forest import RandomForestQuantileRegressor


def find_base_dir(start=None) -> Path:
    """Cari root repo — folder pertama ke atas yang berisi `dataset/csv/`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "csv").is_dir():
            return candidate
    raise RuntimeError(f"Root repo tidak ditemukan dari {start}")


BASE_DIR = find_base_dir()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from utils.modelling import evaluation, model_common, modeling_prep, purging, run_config
from utils.modelling import walk_forward

print(f"BASE_DIR = {BASE_DIR}")

## 2. Definisi model Random Forest kuantil

In [ ]:
# Satu model kandidat lengkap dengan empat tahap pemakaiannya: make_fit_predict() (fungsi
# latih-dan-prediksi yang disuntikkan ke mesin evaluasi), sample_search_space() + run_search()
# (pencarian hyperparameter), fit_final() (pelatihan model akhir), dan predict_bundle()
# (inferensi dari model tersimpan).
#
# Mengapa bukan Random Forest biasa: Random Forest sklearn meminimalkan galat kuadrat atau
# absolut; keduanya menaksir PUSAT distribusi, sehingga ramalannya kehabisan stok kira-kira
# separuh waktu. Quantile regression forest menyimpan seluruh nilai target yang jatuh di tiap
# leaf, lalu membaca kuantil yang diminta dari distribusi empiris tersebut — sehingga satu
# model yang sama dapat menjawab 19 titik kuantil sekaligus.
#
# Konsekuensi biayanya: penyimpanan nilai per-leaf itu berbentuk array padat berukuran
# (n_estimators, max_node_count, n_outputs, max_samples_leaf), sehingga kebutuhan memorinya
# sudah tertentu dari hyperparameter SEBELUM satu pohon pun dibangun — lihat
# estimate_leaf_memory_bytes(). Pohon dalam dengan leaf sangat kecil karenanya tidak terjangkau
# di sini, dan pembatasan itu kebetulan sejalan dengan alasan statistiknya: leaf berisi satu
# sampel tidak dapat menaksir kuantil sama sekali.
QUANTILE = 0.9


QUANTILES = evaluation.QUANTILE_SET_A


MEMORY_BUDGET_BYTES = 3 * 1024 ** 3


IDX_COLS = model_common.IDX_COLS


assert_no_nan = model_common.assert_no_nan


expand_one_hot = model_common.expand_one_hot


select_best = model_common.select_best


load_bundle = model_common.load_bundle


DEFAULT_PARAMS = {
    "n_estimators": 200,
    "max_depth": 16,
    "min_samples_leaf": 50,
    "max_samples_leaf": 20,
    "max_features": "sqrt",
    "max_samples": None,
    "log_target": False,
    "one_hot": False,
    "random_state": 42,
}


SEARCH_SPACE = {
    "max_depth": [12, 16, 20],
    "min_samples_leaf": [20, 50, 100, 200],
    "max_samples_leaf": [1, 20, 50],
    "max_features": ["sqrt", 0.3, 0.5, 1.0],
    "max_samples": [None, 0.5],
    "log_target": [False, True],
    "one_hot": [False, True],
}


ESTIMATOR_KEYS = ("n_estimators", "max_depth", "min_samples_leaf",
                  "max_samples_leaf", "max_features", "max_samples",
                  "random_state")


def estimate_leaf_memory_bytes(params: dict, n_train: int) -> int:
    """Menaksir batas atas memori penyimpanan leaf, dalam byte.

    Dipakai menyaring kandidat sebelum data dimuat, sehingga konfigurasi yang
    tidak terjangkau ditolak tanpa membangun pohon.

        byte = n_estimators x jumlah_node x max_samples_leaf x 8

    Jumlah node dibatasi dua kali: oleh kedalaman, karena pohon berkedalaman d
    memuat paling banyak 2^(d+1) node; dan oleh ukuran leaf, karena n baris
    yang terbagi ke leaf berisi minimal L baris menghasilkan paling banyak
    2n/L node termasuk node internal. Batas yang lebih ketat yang dipakai.
    """
    fraction = params.get("max_samples") or 1.0
    n_bootstrap = n_train * fraction
    depth_bound = 2.0 ** (params["max_depth"] + 1)
    leaf_bound = 2.0 * n_bootstrap / params["min_samples_leaf"]
    node_count = min(depth_bound, leaf_bound)
    return int(params["n_estimators"] * node_count * params["max_samples_leaf"] * 8)


def build_estimator(params: dict) -> RandomForestQuantileRegressor:
    """Menyusun estimator quantile-forest dari satu set hyperparameter."""
    kwargs = {key: params[key] for key in ESTIMATOR_KEYS if key in params}
    return RandomForestQuantileRegressor(n_jobs=-1, **kwargs)

In [ ]:
def make_fit_predict(
    params: Optional[dict] = None,
    feature_cols: Optional[list] = None,
    quantiles: tuple = QUANTILES,
    memory_budget: int = MEMORY_BUDGET_BYTES,
) -> Callable[[pd.DataFrame, pd.DataFrame], np.ndarray]:
    """Membentuk fungsi latih-dan-prediksi yang disuntikkan ke mesin evaluasi.

    Fungsi yang dikembalikan menerima `(train, valid)` dan mengembalikan matriks
    prediksi berukuran `(len(valid), len(quantiles))`. Di dalamnya, berurutan:
    cek NaN, penyaringan budget memori, pemilihan fitur, ekspansi one-hot bila
    diminta, transformasi target, fit, prediksi seluruh grid kuantil, lalu
    pembalikan transformasi dan pemotongan nilai negatif.

    Pemilihan fitur, ekspansi one-hot, dan transformasi target sengaja tinggal
    di sini alih-alih di `walk_forward`, karena ketiganya adalah *pilihan
    model* — persis hal yang seharusnya diperbandingkan antar ketiga kandidat.

    Seluruh grid kuantil keluar dari satu kali fit: leaf sudah memuat distribusi
    training, sehingga setiap titik tambahan hanya berongkos satu pembacaan lagi
    atas distribusi yang memang sudah tersimpan. Itulah sebabnya model ini tidak
    perlu dicari ulang saat kriteria berpindah ke multi-kuantil, sementara dua
    model lainnya perlu.
    """
    params = {**DEFAULT_PARAMS, **(params or {})}
    feature_cols = feature_cols or modeling_prep.FEATURE_COLS
    quantiles = tuple(quantiles)

    def fit_predict(train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
        assert_no_nan(train, feature_cols)
        assert_no_nan(valid, feature_cols)

        needed = estimate_leaf_memory_bytes(params, len(train))
        if needed > memory_budget:
            raise MemoryError(
                f"leaf storage {needed / 1024 ** 3:.1f} GB melebihi budget "
                f"{memory_budget / 1024 ** 3:.1f} GB untuk {params}"
            )

        train_X, valid_X = train[feature_cols], valid[feature_cols]
        if params["one_hot"]:
            train_X, valid_X = expand_one_hot(train_X, valid_X)

        y_train = model_common.train_target(train, log_target=params["log_target"])

        model = build_estimator(params)
        model.fit(train_X.to_numpy(dtype=np.float32), y_train)
        # Satu panggilan, bukan perulangan: mengoper seluruh grid menelusuri
        # leaf sekali, sementara sembilan belas panggilan akan menelusurinya
        # sembilan belas kali untuk jawaban yang sama persis.
        prediction = model.predict(valid_X.to_numpy(dtype=np.float32),
                                   quantiles=list(quantiles))
        prediction = np.asarray(prediction, dtype=float).reshape(len(valid_X),
                                                                 len(quantiles))
        if params["log_target"]:
            prediction = modeling_prep.inverse_log_target(prediction)
        # Kuantitas kirim negatif tidak punya makna fisik.
        return np.clip(prediction, 0.0, None)

    return fit_predict

In [ ]:
SEARCH_FOLDS = (3, 5)


TYPICAL_N_TRAIN = 1_280_000


def sample_search_space(
    n_candidates: int = 18,
    n_train: int = TYPICAL_N_TRAIN,
    seed: int = 42,
    memory_budget: int = MEMORY_BUDGET_BYTES,
    space: Optional[dict] = None,
) -> list:
    """Menarik acak sejumlah kandidat hyperparameter yang unik dan terjangkau.

    Penyaringan budget-lah yang membuat pembungkus ini layak ada: quantile-forest
    menentukan ukuran array nilai per-leaf dari hyperparameter sebelum satu pohon
    pun dibangun, sehingga kandidat yang tidak terjangkau dapat ditolak tanpa
    memuat data sama sekali.
    """
    def screen(candidate: dict) -> bool:
        return estimate_leaf_memory_bytes(candidate, n_train) <= memory_budget

    return model_common.sample_search_space(
        space=SEARCH_SPACE if space is None else space,
        defaults=DEFAULT_PARAMS,
        n_candidates=n_candidates,
        seed=seed,
        screen=screen,
        screen_label=f"budget {memory_budget / 1024 ** 3:.1f} GB",
    )


def run_search(
    df: pd.DataFrame,
    candidates: list,
    folds: tuple = SEARCH_FOLDS,
    quantiles: tuple = QUANTILES,
    model_name: str = "random_forest",
    feature_cols: Optional[list] = None,
    verbose: bool = True,
    checkpoint_path: Optional[str] = None,
    resume: bool = True,
    only: Optional[Iterable[int]] = None,
    provenance: Optional[dict] = None,
) -> pd.DataFrame:
    """Menilai setiap kandidat di fold pencarian, satu baris hasil per kandidat.

    Protokolnya milik `model_common.run_search()` — termasuk checkpoint yang
    ditulis tiap kandidat selesai dan guard yang menolak melanjutkan dari
    checkpoint yang lahir di ruang pencarian atau grid kuantil berbeda —
    sehingga ketiga model dinilai lewat mesin yang sama persis.
    """
    return model_common.run_search(
        df, candidates, make_fit_predict=make_fit_predict,
        search_space=SEARCH_SPACE, folds=folds, quantiles=quantiles,
        model_name=model_name, feature_cols=feature_cols, verbose=verbose,
        checkpoint_path=checkpoint_path, resume=resume,
        only=only, provenance=provenance,
    )

In [ ]:
MODEL_FILE = str(BASE_DIR / "models/random_forest_q90.joblib")


BEST_PARAMS_FILE = str(BASE_DIR / "dataset/model_ready/rf_best_params.json")


FINAL_N_ESTIMATORS = 400


def fit_final(
    df: pd.DataFrame,
    params: dict,
    feature_cols: Optional[list] = None,
    n_estimators: int = FINAL_N_ESTIMATORS,
    quantiles: tuple = QUANTILES,
    date_col: str = modeling_prep.DATE_COL,
    test_start: pd.Timestamp = modeling_prep.TEST_START,
) -> dict:
    """Melatih model akhir pada seluruh baris layak sebelum Desember.

    Mengembalikan *bundle*: model terlatih beserta hyperparameter, daftar dan
    urutan kolom training, grid kuantil, jumlah baris, dan provenance target —
    segala yang dibutuhkan `predict_bundle()` untuk mengulang perlakuan yang
    sama saat inferensi.

    Kelayakan baris datang dari `walk_forward.eligible_rows()`, bukan dari
    penyaringan tanggal yang ditulis di sini. Baris yang melatih model akhir
    harus sama dengan baris yang menilainya, dan pemotongan saat penilaian
    bukan hanya soal tanggal: 28 hari pertama tiap segmen belum punya jendela
    lag yang penuh, dan beberapa hari terakhir tidak punya target sama sekali
    karena penjumlahan lead-time melewati ujung data. Menyeleksi baris secara
    terpisah di sini pernah membuat model yang dikirim dilatih pada populasi
    yang berbeda dari yang dilaporkan metriknya — dan, karena pemotongan target
    ikut hilang, pada label yang bernilai NaN.

    `purging.lookahead_safe_mask()` memotong di batas Desember supaya tidak ada
    baris training yang targetnya menjangkau ke dalam test set.

    Bundle mencatat urutan kolom training bersama modelnya. Forest yang dimuat
    ulang pekan depan dengan urutan kolom berbeda tidak gagal — ia meramal
    dengan percaya diri dari fitur yang salah, dan itu lebih buruk.
    """
    params = {**DEFAULT_PARAMS, **params, "n_estimators": n_estimators}
    feature_cols = feature_cols or modeling_prep.FEATURE_COLS

    frame = walk_forward.eligible_rows(df, date_col=date_col, test_start=test_start)
    frame = frame[purging.lookahead_safe_mask(frame, test_start, date_col=date_col)]
    assert_no_nan(frame, feature_cols)

    train_X = frame[feature_cols]
    if params["one_hot"]:
        train_X, _ = expand_one_hot(train_X, train_X)

    y_train = model_common.train_target(frame, log_target=params["log_target"])

    model = build_estimator(params)
    model.fit(train_X.to_numpy(dtype=np.float32), y_train)
    return {
        "model": model,
        "params": params,
        "feature_cols": feature_cols,
        "columns": list(train_X.columns),
        "quantiles": tuple(quantiles),
        "n_train": int(len(frame)),
        **model_common.target_provenance(),
    }


def predict_bundle(bundle: dict, frame: pd.DataFrame) -> np.ndarray:
    """Meramal dengan bundle terlatih, memaksa urutan kolom yang tercatat.

    Grid kuantilnya dibaca dari bundle, bukan dari konstanta modul ini. Forest
    yang dimuat ulang setelah peralihan Tahap A -> Tahap B harus menjawab di
    titik-titik tempat ia dilaporkan, bukan di grid apa pun yang sedang berlaku.
    """
    params = bundle["params"]
    quantiles = tuple(bundle["quantiles"])
    features = frame[bundle["feature_cols"]]
    if params["one_hot"]:
        features, _ = expand_one_hot(features, features)
    features = features.reindex(columns=bundle["columns"], fill_value=0)
    prediction = bundle["model"].predict(
        features.to_numpy(dtype=np.float32), quantiles=list(quantiles)
    )
    prediction = np.asarray(prediction, dtype=float).reshape(len(features),
                                                             len(quantiles))
    if params["log_target"]:
        prediction = modeling_prep.inverse_log_target(prediction)
    return np.clip(prediction, 0.0, None)


def save_bundle(bundle: dict, path: str = MODEL_FILE) -> None:
    """Menyimpan bundle model akhir ke berkas joblib."""
    model_common.save_bundle(bundle, path)


def save_best_params(params: dict, path: str = BEST_PARAMS_FILE) -> None:
    """Menyimpan hyperparameter pemenang pencarian ke berkas JSON."""
    model_common.save_best_params(params, path)

## 3. Setelan run & data model-ready

In [ ]:
# Sel persiapan: memuat panel model-ready dan membaca setelan run dari
# environment (FORECAST_*), sehingga notebook yang sama dapat dijalankan di
# mesin yang berbeda tanpa diedit. Tanpa satu pun env var, perilakunya adalah
# default — seluruh kandidat, device "cpu", path artefak bawaan.
#
# Random Forest tidak punya jalur GPU: `quantile-forest` murni CPU, jadi
# FORECAST_DEVICE tidak berarti di sini dan modelnya selalu berjalan di CPU
# (Bagian 2 spec eksekusi terdistribusi). Setelan lain tetap dibaca supaya
# ketiga notebook berperilaku seragam dan `device` tercatat "cpu" di tiap
# baris hasil, bukan kosong.
DEVICE = run_config.device("cpu")
SHARD = run_config.shard()
SEARCH_FILE = run_config.search_checkpoint("rf")
RESULTS_FILE = run_config.checkpoint_path("rf_walk_forward_results.csv")

df = pd.read_parquet(run_config.model_input_path())
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(run_config.describe(DEVICE))

# Path artefak datang dari run_config, bukan dari konstanta modul, supaya FORECAST_CHECKPOINT_DIR
# berlaku untuk seluruh sel — termasuk §7 yang membaca ulang tabel hasil. Tanpa env var apa pun
# keduanya menunjuk berkas yang sama.
print(f"SEARCH_FILE  = {SEARCH_FILE}")
print(f"RESULTS_FILE = {RESULTS_FILE}")

## 4. Benchmark

In [2]:
# Satu kali fit pada training set penuh fold 5 dengan DEFAULT_PARAMS.
#
# Tujuannya bukan mencetak skor, melainkan menjawab dua pertanyaan sebelum 18 fit dijalankan:
# apakah taksiran memori leaf yang dipakai menyaring kandidat memang berlaku, dan berapa lama
# satu fit sebenarnya berjalan. Angkanya dicatat di docs/hasil-modeling-rf.md §3.
import resource
import time

# Menyiapkan fold 5, menaksir kebutuhan memori leaf dari hyperparameter, lalu
# menjalankan satu fit + prediksi sambil mencatat wall time dan puncak RSS.
split = walk_forward.prepare_fold(df, 5)
train, valid = split["train"], split["valid"]
print(f"train {len(train):,} rows, valid {len(valid):,} rows")
print(f"QUANTILE_SET: {len(QUANTILES)} titik, {QUANTILES[0]}..{QUANTILES[-1]}")

params = dict(DEFAULT_PARAMS)
print("estimated leaf storage: "
      f"{estimate_leaf_memory_bytes(params, len(train)) / 1024 ** 3:.2f} GB")

start = time.time()
prediction = make_fit_predict(params)(train, valid)
elapsed = time.time() - start

peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # byte di macOS
# Indeks titik kuantil terdekat ke tau=0,9, yaitu angka yang dijanjikan ke
# bisnis (B-9); dipakai hanya untuk mencetak ringkasan yang mudah dibaca.
headline = min(range(len(QUANTILES)),
               key=lambda i: abs(QUANTILES[i] - evaluation.DEFAULT_ALPHA))
print(f"wall time {elapsed / 60:.1f} min")
print(f"peak RSS  {peak_bytes / 1024 ** 3:.2f} GB")
print(f"prediction shape {prediction.shape} (baris x titik kuantil)")
print(f"di tau=0.9: mean {prediction[:, headline].mean():.2f}, "
      f"max {prediction[:, headline].max():.2f}")
# Nol secara struktural: setiap titik adalah persentil dari satu distribusi
# leaf yang sama, jadi inversi mustahil. Dicetak sebagai bukti, bukan harapan —
# nilai bukan-nol di sini berarti ada bug, bukan model yang lemah.
print(f"crossing_rate {evaluation.crossing_rate(prediction, QUANTILES):.4f}")

train 1,292,778 rows, valid 59,629 rows
QUANTILE_SET: 19 titik, 0.05..0.95
estimated leaf storage: 1.54 GB
wall time 9.7 min
peak RSS  4.14 GB
prediction shape (59629, 19) (baris x titik kuantil)
di tau=0.9: mean 47.66, max 1772.10
crossing_rate 0.0000


## 5. Pencarian hyperparameter

In [3]:
# 18 kandidat ditarik acak dari ruang 1.152 kombinasi (seed 42), disaring lebih dulu terhadap
# budget memori 3 GB, lalu dinilai di FOLD 3 DAN 5 dengan kriteria K1 — rata-rata pinball
# lintas 19 titik kuantil.
#
# Dua fold, bukan lima: pencarian bertugas MEMERINGKAT kandidat, bukan melaporkan hasil. Angka
# yang dilaporkan datang dari walk-forward di §6, karena menilai di fold yang ikut memilih
# pemenang bersifat optimistis.
# Menilai ke-18 kandidat di fold pencarian, lalu memilih pemenangnya. Hasil
# tiap kandidat ditulis ke checkpoint begitu selesai, sehingga run yang
# terputus dapat dilanjutkan tanpa mengulang kandidat yang sudah dinilai.
#
# Catatan butir 0c (keputusan pemilik proyek 2026-08-24): pencarian ini
# **dijalankan ulang**, membalik revisi sebelumnya. Alasan revisi lama tidak
# dibantah — hyperparameter forest membentuk *leaf*, dan seluruh titik
# QUANTILE_SET dibaca dari leaf yang sama, jadi migrasi multi-kuantil sendiri
# tidak mengubah pertanyaan yang dijawab pencarian ini. Yang membalikkannya
# adalah datanya: rf_best_params.json lama dipilih 2026-08-18, sebelum
# reklasifikasi WIP-2 masuk ke artefak (dibangun ulang 2026-08-23 22:52) —
# kebasian yang sama yang dipakai sebagai alasan membuang bundle terlatihnya.
# Anggaran tidak berubah: 18 kandidat, SEARCH_FOLDS = (3, 5), seed 42.
# Lihat 2026-08-18-random-forest-modeling-design.md Part 2.
#
# Prasyarat langkah 0 Fase 3 sudah dipenuhi: rf_search_results.csv DAN
# rf_best_params.json dari run kuantil-tunggal diganti nama menjadi
# `*.single-quantile.bak.*` (2026-08-24) sesudah guard checkpoint diverifikasi
# berbunyi — RF berhenti setelah 2,8 detik. Kalau berkas tanpa kolom
# `headline_quantile` muncul lagi di jalur checkpoint, guard yang sama di
# model_common._assert_checkpoint_matches() akan menolaknya lagi.
train_size = len(walk_forward.prepare_fold(df, 5)["train"])
candidates = sample_search_space(18, n_train=train_size, seed=42)
search_results = run_search(df, candidates, folds=SEARCH_FOLDS,
                               checkpoint_path=SEARCH_FILE, only=SHARD,
                               provenance=run_config.provenance(DEVICE))
search_results.to_csv(SEARCH_FILE, index=False)

# Pemenang dipilih di sel ini, jadi guard shard-nya juga di sini: memilih
# dari sebagian kandidat menghasilkan angka yang tampak sepenuhnya wajar.
assert SHARD is None, (
    "run bershard: jangan pilih pemenang dari sebagian kandidat — "
    "gabungkan seluruh shard dengan model_common.merge_shards() lebih dulu"
)
best = select_best(search_results, candidates)

print(best)

[1/18] pinball=2.9228 epoch=- 602s 
[2/18] pinball=2.8808 epoch=- 1234s 
[3/18] pinball=3.1969 epoch=- 628s 
[4/18] pinball=2.9940 epoch=- 902s 
[5/18] pinball=2.9404 epoch=- 324s 
[6/18] pinball=3.0509 epoch=- 324s 
[7/18] pinball=2.9320 epoch=- 559s 
[8/18] pinball=2.9232 epoch=- 276s 
[9/18] pinball=2.9764 epoch=- 791s 
[10/18] pinball=2.9108 epoch=- 1118s 
[11/18] pinball=3.1838 epoch=- 444s 
[12/18] pinball=2.9832 epoch=- 1540s 
[13/18] pinball=2.9560 epoch=- 745s 
[14/18] pinball=3.0003 epoch=- 724s 
[15/18] pinball=3.0160 epoch=- 701s 
[16/18] pinball=2.9785 epoch=- 889s 
[17/18] pinball=2.9165 epoch=- 1014s 
[18/18] pinball=2.8984 epoch=- 1029s 
{'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 20, 'max_samples_leaf': 1, 'max_features': 1.0, 'max_samples': None, 'log_target': False, 'one_hot': False, 'random_state': 42}


## 6. Walk-forward lima fold

In [4]:
# Konfigurasi pemenang dinilai ulang di kelima fold (validasi Juli-November 2025), melawan tiga
# baseline naive pada baris yang IDENTIK — dijamin oleh walk_forward.run_walk_forward(), yang
# memiliki definisi fold dan kelayakan baris, serta menerima model sebagai fungsi yang
# disuntikkan.
#
# Fold 3 dan 5 ikut memilih pemenang, sehingga skornya di sana bukan out-of-sample terhadap
# seleksi. Potongan fold 1, 2, dan 4 adalah yang bersih, dan potongan itulah yang menjadi K1
# resmi.
# Menyimpan hyperparameter pemenang, menjalankan walk-forward lima fold, lalu
# menulis tabel hasilnya ke CSV — satu baris per (model x fold x kuantil x
# potongan). Tabel yang dicetak di bawah adalah pinball di tau=0,9 per fold.
save_best_params(best)

fit_predict = make_fit_predict(best)
results = walk_forward.run_walk_forward(df, fit_predict, model_name="random_forest",
                                        quantiles=QUANTILES)
results.to_csv(RESULTS_FILE, index=False)

print(f"K1 (rata-rata pinball lintas {len(QUANTILES)} kuantil): "
      f"{walk_forward.pooled_k1(results, 'random_forest'):.4f}")

overall = results[results["group_col"].isna()]
(overall[(overall["quantile"] - evaluation.DEFAULT_ALPHA).abs() < 1e-9]
 .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

K1 (rata-rata pinball lintas 19 kuantil): 2.8621


fold_id,1,2,3,4,5
model,,,,,
naive_lag_1,8.355,8.469,8.045,8.526,8.372
naive_roll_mean_7,4.249,4.566,4.034,4.783,4.970
naive_zero,26.453,27.254,23.849,26.219,29.320
random_forest,2.263,2.440,2.402,2.757,2.551


## 7. Model final

In [5]:
# Konfigurasi pemenang dilatih ulang pada SELURUH baris layak sebelum Desember — populasi yang
# sama persis dengan yang dinilai di atas — dengan jumlah pohon dinaikkan 200 -> 400.
#
# Hasilnya disimpan sebagai bundle: model beserta urutan kolom training, grid kuantil, dan
# provenance target, sehingga inferensi di kemudian hari mengulang perlakuan yang sama dan
# tidak diam-diam meramal dari kolom yang salah.
# Melatih model akhir pada seluruh baris layak sebelum Desember, lalu menyimpan
# bundle-nya ke models/random_forest_q90.joblib.
bundle = fit_final(df, best)
save_bundle(bundle)
print(f"trained on {bundle['n_train']:,} rows, "
      f"{len(bundle['columns'])} columns, "
      f"{len(bundle['quantiles'])} titik kuantil "
      f"({bundle['quantiles'][0]}..{bundle['quantiles'][-1]})")

trained on 1,349,011 rows, 56 columns, 19 titik kuantil (0.05..0.95)


## 8. Ringkasan hasil

In [6]:
# Beberapa potongan, masing-masing melawan ketiga baseline pada baris identik: satu angka global
# menyesatkan di data yang 44% targetnya bernilai nol.
#
# Yang dicetak berurutan: K1 gabungan per model, K1 per fold, pinball di tau=0,9 per fold (angka
# headline B-9), K1 per demand_segment dan per is_delivery_day, kalibrasi di tau=0,9, lalu K2 —
# coverage di seluruh 19 titik kuantil.
#
# Pembacaan lengkapnya di docs/hasil-modeling-rf.md; khususnya §5.2, yang menjelaskan mengapa
# kolom `gap` pada tabel K2 terakhir belum boleh dibaca sebagai kalibrasi di tau rendah.
# Membaca ulang tabel hasil dari CSV (supaya sel ini dapat dijalankan sendiri
# tanpa mengulang walk-forward), lalu mencetak potongan-potongan laporannya.
results = pd.read_csv(RESULTS_FILE)
HEADLINE = evaluation.DEFAULT_ALPHA

print("=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} K1 {walk_forward.pooled_k1(results, model):7.4f}")

print("\n=== per fold, K1 ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

print(f"\n=== per fold, pinball di tau={HEADLINE} (angka headline B-9) ===")
headline_rows = results[results["group_col"].isna()
                        & ((results["quantile"] - HEADLINE).abs() < 1e-9)]
print(headline_rows.pivot_table(index="model", columns="fold_id",
                                values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (K1, pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    # Kolom dipilih sebelum apply: tanpa itu pandas ikut menyertakan kolom
    # pengelompokan dan mengeluarkan FutureWarning di setiap sel.
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)[["weighted", "n"]]
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

# pooled_metric menolak merata-ratakan lintas kuantil untuk metrik selain
# pinball/crossing_rate — coverage di 0,05 dan di 0,95 menjawab pertanyaan yang
# berbeda. Jadi ketiganya dibaca di tau headline saja, secara eksplisit.
print(f"\n=== coverage / fill rate di tau={HEADLINE} (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage', quantile=HEADLINE):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate', quantile=HEADLINE):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units', quantile=HEADLINE):9.1f}  "
          f"crossing {walk_forward.pooled_metric(results, model, 'crossing_rate'):6.4f}")

# Kolom `gap` di bawah adalah coverage - tau. Di tau rendah ia belum mengukur
# kalibrasi: 41,95% baris validasi bertarget nol dan `0 <= 0` selalu tercakup,
# sehingga ada lantai coverage yang tidak dapat ditembus model tak-negatif mana
# pun. Lihat docs/hasil-modeling-md Bagian 5.2.
print("\n=== K2: coverage per titik kuantil (random_forest) ===")
print(walk_forward.coverage_by_quantile(results, "random_forest").round(4)
      .to_string(index=False))

=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===
random_forest        K1  2.8621
naive_zero           K1 14.7469
naive_lag_1          K1  8.1755
naive_roll_mean_7    K1  4.8231

=== per fold, K1 ===
fold_id                 1       2       3       4       5
model                                                    
naive_lag_1         7.955   8.533   8.108   7.971   8.307
naive_roll_mean_7   4.664   4.981   4.668   4.937   4.872
naive_zero         14.696  15.141  13.249  14.566  16.289
random_forest       2.682   2.844   2.754   3.040   3.031

=== per fold, pinball di tau=0.9 (angka headline B-9) ===
fold_id                 1       2       3       4       5
model                                                    
naive_lag_1         8.355   8.469   8.045   8.526   8.372
naive_roll_mean_7   4.249   4.566   4.034   4.783   4.970
naive_zero         26.453  27.254  23.849  26.219  29.320
random_forest       2.263   2.440   2.402   2.757   2.551

=== per demand_segment (K1, pooled 

## 9. *(Opsional)* Cek sinkron dengan `utils/`

In [ ]:
import inspect

from utils.modelling import model_random_forest as _ref

# SEARCH_FILE dan RESULTS_FILE sengaja datang dari run_config di §3, jadi nilainya boleh berbeda
# dari konstanta modul begitu FORECAST_CHECKPOINT_DIR diset. Sisanya harus identik.
_LEWATI = {"find_base_dir", "SEARCH_FILE", "RESULTS_FILE",
           "_LEWATI", "_ref", "_beda", "_n_fungsi", "_n_konstanta"}

_beda, _n_fungsi, _n_konstanta = [], 0, 0
for nama, obj in sorted(globals().items()):
    if nama in _LEWATI or nama.startswith("__") or not hasattr(_ref, nama):
        continue
    ref = getattr(_ref, nama)
    if inspect.isfunction(obj):
        _n_fungsi += 1
        if inspect.getsource(obj) != inspect.getsource(ref):
            _beda.append(nama)
    elif nama.isupper():
        _n_konstanta += 1
        if obj != ref:
            _beda.append(nama)

if _beda:
    print("BERBEDA dari utils/ — salin ulang atau samakan: " + ", ".join(_beda))
else:
    print(f"Sinkron: {_n_fungsi} fungsi + {_n_konstanta} konstanta identik dengan "
          f"utils/modelling/model_random_forest.py")